# Second evaluation: DR25 KOI table

Scores the trained model on Kepler Objects of Interest (KOIs) from the Q1–Q17 DR25 KOI table.

- **Unseen stars only:** every star in `train.npz` (train + val) is dropped, so the model has never seen any of these light curves.
- **Labels:** `koi_disposition` CONFIRMED = 1, FALSE POSITIVE = 0. CANDIDATE KOIs have no answer yet, so they are scored but not used in the metrics.
- **Harder than the DR24 test set:** KOIs were already vetted as transit-like, so the easy noise (`NTP`) negatives are gone.

Run on a Colab GPU runtime. Upload the KOI csv to the same Drive folder as `train.npz` / `test.npz` / `models/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/exosalience_data"
KOI_CSV = f"{DRIVE}/q1_q17_dr25_koi_2026.11.17_11.59.52.csv"

In [ ]:
!test -d /content/ExoSalience || git clone -b karen https://github.com/03karenxu/ExoSalience.git /content/ExoSalience
%cd /content/ExoSalience
!pip install -q lightkurve s3fs

## Pick the evaluation KOIs

In [ ]:
import numpy as np
import pandas as pd

kois = pd.read_csv(KOI_CSV, comment="#")
train_stars = set(np.load(f"{DRIVE}/train.npz")["kepid"])  # train + val: the model has seen these stars
test_stars = set(np.load(f"{DRIVE}/test.npz")["kepid"])

kois = kois[~kois.kepid.isin(train_stars)].reset_index(drop=True)
kois["group"] = np.where(kois.kepid.isin(test_stars), "DR24 test star", "new star")

# preprocess() expects the TCE catalogue's column names
kois = kois.rename(columns={"koi_period": "tce_period", "koi_time0bk": "tce_time0bk",
                            "koi_duration": "tce_duration", "koi_tce_plnt_num": "tce_plnt_num"})

print(len(kois), "KOIs on", kois.kepid.nunique(), "unseen stars")
pd.crosstab(kois.koi_disposition, kois.group, margins=True)

## Build global / local views

Same preprocessing as training, streamed from the MAST S3 bucket. Slow (thousands of stars), so views are saved to Drive in chunks: if the session drops, rerun this cell and it skips finished chunks. Set `LIMIT` to a small number for a quick trial run first.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

from src.preprocessing.load import from_s3
from src.preprocessing.preprocess import preprocess

CACHE = Path(DRIVE) / "koi_views"
CACHE.mkdir(exist_ok=True)
CHUNK = 200    # stars per saved chunk
LIMIT = None   # e.g. 50 for a quick trial run

def views_for_star(star):
    """(kepoi_name, global_view, local_view) per KOI on the star, plus an error message if it failed"""
    kepid, star_kois = star
    try:
        views = preprocess(kepid, from_s3(kepid), star_kois)  # views come back in star_kois row order
        return [(name, g, l) for name, (_, g, l) in zip(star_kois.kepoi_name, views)], None
    except Exception as e:
        return [], f"{kepid}: {type(e).__name__}: {e}"

stars = list(kois.groupby("kepid"))[:LIMIT]
with ProcessPoolExecutor(max_workers=8) as pool:  # mostly waiting on S3, so more workers than CPUs is fine
    for i in range(0, len(stars), CHUNK):
        path = CACHE / f"chunk_{i // CHUNK:03d}.npz"
        if path.exists():
            continue
        results = list(pool.map(views_for_star, stars[i:i + CHUNK]))
        rows = [r for rs, _ in results for r in rs]
        errors = [err for _, err in results if err]
        np.savez_compressed(path,
                            kepoi_name=np.array([n for n, _, _ in rows], dtype=str),
                            global_view=np.array([g for _, g, _ in rows]).reshape(-1, 2001),
                            local_view=np.array([l for _, _, l in rows]).reshape(-1, 201))
        print(f"chunk {i // CHUNK}: {len(rows)} KOIs, {len(errors)} stars failed", *errors[:3], sep="\n  ")

## Score with the trained model

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.jit.load(f"{DRIVE}/models/model_scripted.pt", map_location=device).eval()
threshold = torch.load(f"{DRIVE}/models/checkpoints/model_final.pt", map_location="cpu")["config"]["threshold"]
print("device:", device, "| cutoff picked on DR24 val:", round(threshold, 3))

chunks = [np.load(p) for p in sorted(CACHE.glob("chunk_*.npz"))]
names = np.concatenate([c["kepoi_name"] for c in chunks])
xg = torch.from_numpy(np.concatenate([c["global_view"] for c in chunks])).float().unsqueeze(1)
xl = torch.from_numpy(np.concatenate([c["local_view"] for c in chunks])).float().unsqueeze(1)

with torch.no_grad():
    probs = torch.cat([
        torch.sigmoid(model(xg[i:i + 512].to(device), xl[i:i + 512].to(device), None)).squeeze(1).cpu()
        for i in range(0, len(xg), 512)
    ]).numpy()

res = kois.merge(pd.DataFrame({"kepoi_name": names, "prob": probs}), on="kepoi_name")
res["pred"] = res.prob >= threshold
print(len(res), "of", len(kois), "KOIs scored")

## Metrics: CONFIRMED vs FALSE POSITIVE

Same metrics as `train.py`, at the cutoff picked on the DR24 validation set. Split by group: **DR24 test star** KOIs overlap the DR24 test set (same stars, NASA's labels instead of Autovetter's); **new star** KOIs are on stars that had no labelled TCE in DR24 at all.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

def report(df, name):
    y = (df.koi_disposition == "CONFIRMED").astype(int)
    tn, fp, fn, tp = confusion_matrix(y, df.pred, labels=[0, 1]).ravel()
    print(f"{name:15s} n={len(df):4d} | roc_auc={roc_auc_score(y, df.prob):.4f} "
          f"prec={precision_score(y, df.pred):.4f} recall={recall_score(y, df.pred):.4f} f1={f1_score(y, df.pred):.4f} "
          f"| tp={tp} fp={fp} fn={fn} tn={tn}")

known = res[res.koi_disposition != "CANDIDATE"]
report(known, "all")
for group, df in known.groupby("group"):
    report(df, group)

For reference: NASA's own Robovetter score (`koi_score`) on the same KOIs. Not a fair contest: the Robovetter also uses centroid motion and other data the model never sees, and the dispositions were partly decided with it.

In [ ]:
scored = known.dropna(subset=["koi_score"])
print(f"Robovetter koi_score roc_auc={roc_auc_score(scored.koi_disposition == 'CONFIRMED', scored.koi_score):.4f} (n={len(scored)})")

## Which false positives fool the model?

The KOI table flags why each false positive was rejected (a KOI can have several flags). The model only sees light-curve shape, so expect it to miss **centroid offset** and **ephemeris match** false positives (the dip is real but comes from another star) much more often than **stellar eclipse** ones.

In [ ]:
fps = known[known.koi_disposition == "FALSE POSITIVE"]
flags = {"koi_fpflag_nt": "not transit-like", "koi_fpflag_ss": "stellar eclipse",
         "koi_fpflag_co": "centroid offset", "koi_fpflag_ec": "ephemeris match"}
for flag, desc in flags.items():
    sub = fps[fps[flag] == 1]
    print(f"{desc:17s} n={len(sub):4d} | wrongly called planet: {sub.pred.mean():.1%}")

## Unresolved candidates

In [ ]:
cands = res[res.koi_disposition == "CANDIDATE"]
print(f"{cands.pred.mean():.1%} of {len(cands)} CANDIDATE KOIs predicted planet")
cands.sort_values("prob", ascending=False)[["kepoi_name", "prob", "koi_score", "koi_prad", "tce_period"]].head(20)

In [ ]:
# save predictions for the frontend
res[["kepid", "kepoi_name", "kepler_name", "koi_disposition", "group", "prob", "pred"]].to_csv(f"{DRIVE}/koi_predictions.csv", index=False)